In [ ]:
#| include: false
import timm
from fastai.vision.all import *
from fasterai.quantize.quantizer import *

Skipping import of cpp extensions due to incompatible torch version 2.9.1+cu128 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info


In [ ]:
path = untar_data(URLs.PETS)
files = get_image_files(path/"images")

def label_func(f): return f[0].isupper()

dls = ImageDataLoaders.from_name_func(path, files, label_func, item_tfms=Resize(64))

In [ ]:
pretrained_resnet_34 = timm.create_model('resnet34', pretrained=True)
learn = Learner(dls, pretrained_resnet_34, metrics=accuracy)
learn.model.fc = nn.Linear(512, 2)
learn.fit_one_cycle(5, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,0.524684,0.393702,0.811908,00:02
1,0.308526,0.307400,0.871448,00:02
2,0.196234,0.314265,0.862652,00:02
3,0.103283,0.222099,0.914750,00:02
4,0.076600,0.215802,0.914750,00:02


In [ ]:
quantizer = Quantizer(
    backend="x86",
    method="static",    # Use dynamic quantization
    verbose=True,       # See detailed output for debugging
    use_per_tensor=False
)

# Quantize your model
quantized_model = quantizer.quantize(
    model=learn.model,
    calibration_dl=dls.train,
    max_calibration_samples= 200,
)

Preparing model for static quantization with x86 backend


/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torch/ao/quantization/observer.py:246: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(


Calibrating with up to 200 samples


Calibrating: 100%|███████████████████████████████████████████| 3/3 [00:01<00:00,  2.62it/s]


Converting to quantized model
Quantization complete


In [ ]:
#| include: false
from tqdm import tqdm

def get_model_size(model):
    torch.save(model.state_dict(), "temp.p")
    size = os.path.getsize("temp.p") / 1e6  # Size in MB
    os.remove("temp.p")
    return size
    
def compute_validation_accuracy(model, valid_dataloader, device=None):
    # Set the model to evaluation mode
    model.eval()
    
    # Use the model's device if no device is specified
    
    device = torch.device('cpu')
    
    # Move model to the specified device
    model = model.to(device)
    
    # Tracking correct predictions and total samples
    total_correct = 0
    total_samples = 0
    
    # Disable gradient computation for efficiency
    with torch.no_grad():
        for batch in tqdm(valid_dataloader):
            # Assuming batch is a tuple of (inputs, labels)
            # Adjust this if your dataloader returns a different format
            inputs, labels = batch
            
            # Move inputs and labels to the same device as the model
            inputs = torch.Tensor(inputs).to(device)
            labels = labels.to(device)
            
            # Forward pass
            outputs = model(inputs)
            
            # Get predictions (for classification tasks)
            # Use argmax along the class dimension
            _, predicted = torch.max(outputs, 1)
            
            # Update counters
            total_samples += labels.size(0)
            total_correct += (predicted == labels).sum().item()
    
    # Compute accuracy as a percentage
    accuracy = (total_correct / total_samples) * 100
    
    return accuracy

In [ ]:
print(f'Size of the original model: {get_model_size(learn.model):.2f} MB')
print(f'Size of the quantized model: {get_model_size(quantized_model):.2f} MB')

Size of the original model: 85.27 MB
Size of the quantized model: 21.51 MB


In [ ]:
compute_validation_accuracy(quantized_model, dls.valid)

100%|██████████████████████████████████████████████████████| 24/24 [00:01<00:00, 16.87it/s]


91.13667117726658

---

## torchao Backend

The `torchao` backend uses PyTorch's modern quantization library. It supports INT4 and INT8 weight-only quantization, primarily targeting **Linear layers** (transformers, MLPs). No calibration data needed.

### When to use torchao vs legacy

| | Legacy (`x86`, `qnnpack`) | torchao |
|--|--------------------------|---------|
| **Architecture** | CNNs (Conv2d + Linear) | Transformers, MLPs (Linear) |
| **Bit widths** | INT8 only | INT4, INT8 |
| **Calibration** | Required (static) | Not required (weight-only) |
| **Output device** | CPU only | CPU or GPU |
| **Best for** | Production CNN deployment | Modern model compression |

### INT8 Weight-Only Quantization

The simplest torchao method — no calibration data needed:

In [ ]:
import torch
import torch.nn as nn
from fasterai.quantize.quantizer import Quantizer

# Create a simple transformer-style model
model = nn.Sequential(
    nn.LayerNorm(256),
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.LayerNorm(512),
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 10),
)

# INT8 weight-only — one line, no calibration
quantized = Quantizer(backend='torchao', method='int8_weight_only').quantize(model)

# Verify it works
x = torch.randn(4, 256)
print(f"Output shape: {quantized(x).shape}")
print(f"Output finite: {torch.isfinite(quantized(x)).all()}")

Output shape: torch.Size([4, 10])
Output finite: True


### INT4 Weight-Only Quantization

Maximum compression — 4x smaller weights than INT8. Best with `torch.compile` for GPU speedup:

In [ ]:
# INT4 weight-only — maximum compression
quantized_int4 = Quantizer(backend='torchao', method='int4_weight_only').quantize(model)

print(f"Output shape: {quantized_int4(x).shape}")
print(f"Output finite: {torch.isfinite(quantized_int4(x)).all()}")

# For GPU speedup, combine with torch.compile:
# compiled = torch.compile(quantized_int4, mode='max-autotune')

/home/nathan/miniconda3/envs/dev/lib/python3.12/site-packages/torchao/quantization/quant_api.py:1825: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


ImportError: torchao method 'int4_weight_only' requires additional dependencies: Requires mslk >= 1.0.0

### On a Real Transformer

torchao works directly with PyTorch's `nn.TransformerEncoder`:

In [ ]:
# Build a transformer
encoder_layer = nn.TransformerEncoderLayer(d_model=256, nhead=8, dim_feedforward=512, batch_first=True)
transformer = nn.TransformerEncoder(encoder_layer, num_layers=4).eval()

# Quantize all Linear layers (FFN + attention projections)
transformer_q = Quantizer(backend='torchao', method='int8_weight_only').quantize(transformer)

# Check it works
x = torch.randn(2, 10, 256)  # (batch, sequence, features)
out = transformer_q(x)
print(f"Input:  {x.shape}")
print(f"Output: {out.shape}")
print(f"Finite: {torch.isfinite(out).all()}")

---

## Summary

| Method | Backend | Calibration | Best For |
|--------|---------|-------------|----------|
| `static` | `x86`/`qnnpack` | Required | CNNs (Conv2d + Linear) |
| `dynamic` | `x86`/`qnnpack` | No | RNNs, quick experiments |
| `int8_weight_only` | `torchao` | No | Transformers, MLPs |
| `int4_weight_only` | `torchao` | No | Maximum compression (GPU) |
| `int8_dynamic` | `torchao` | No | Activation + weight quant |

---

## See Also

- [Quantizer API](../../quantize/quantizer.html) - Full API reference
- [QuantizeCallback Tutorial](tutorial.quantize_callback.html) - Quantization-aware training
- [BN Folding](../../misc/bn_folding.html) - Fold BatchNorm before quantization (CNNs)